In [6]:
from math import sqrt
import torch as th
import torch.nn as nn
from torch.nn.functional import softmax

def attentionSDP(queries, keys, values, mask=None):
    """scaled dot-product attention
    """
    
    scaled = queries.matmul(keys.transpose(-2, -1))/sqrt(len(keys))

    if mask is not None:
        scaled = scaled.masked_fill(mask == 0, -1e9)

    return softmax(scaled).matmul(values)

def attentionSDP_oneline(queries, keys, values):
    return softmax(queries.matmul(keys.transpose(-2, -1))/sqrt(len(keys))).matmul(values)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()

        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // num_heads

        self.q_linear = nn.Linear(self.d_model, self.d_model)
        self.k_linear = nn.Linear(self.d_model, self.d_model)
        self.v_linear = nn.Linear(self.d_model, self.d_model)
        self.out_linear = nn.Linear(self.d_model, self.d_model)

    def forward(self, queries, keys, values, mask=None):
        # TODO: apparently need to split up/reshape the tensors from having a d_model dim to two (n_heads, d_k) dims
        
        queries = self.q_linear(queries)
        keys = self.k_linear(keys)
        values = self.v_linear(values)

        values = attentionSDP(queries, keys, values, mask=mask)

        # TODO: here we apparently then need to "unsplit" values?
        return self.out_linear(values)